In [1]:
!pip install -q transformers datasets evaluate accelerate scikit-learn huggingface_hub torch

import os
import time
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
import evaluate
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from huggingface_hub import login

# Verificación de entorno de ejecución GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de ejecución asignado: {device}")
if device == "cuda":
    print(f"GPU instalada: {torch.cuda.get_device_name(0)}")

HF_TOKEN = TOKENHF
if HF_TOKEN != TOKENHF:
    login(token=HF_TOKEN)

else:
    print("Aviso: Autenticación omitida o token por defecto.")

print("TOKEN AUTENTICADO")

Dispositivo de ejecución asignado: cuda
GPU instalada: NVIDIA A100-SXM4-40GB
Aviso: Autenticación omitida o token por defecto.
TOKEN AUTENTICADO


In [2]:
import os
import time
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
import evaluate
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from huggingface_hub import login

raw_data = {
    "text": [
        "El motor presenta fugas de aceite en el sello principal",
        "Error de autenticación al intentar iniciar sesión en el software de diagnóstico",
        "La presión hidráulica cae rápidamente durante la operación en carga",
        "No se puede exportar el reporte en formato PDF desde el panel",
        "Ruido inusual y sobrecalentamiento en la bomba hidrostática",
        "Requerimos restablecer la contraseña del usuario administrador",
        "El sensor de temperatura marca lecturas erráticas constantemente",
        "Fallo de conexión a la base de datos central de mantenimiento",
        "La banda de transmisión muestra desgaste prematuro y grietas",
        "La interfaz web no carga la vista de monitoreo en tiempo real",
        "Fuga de refrigerante detectada cerca del radiador auxiliar",
        "El sistema despliega un código de error 500 al guardar cambios",
        "Vibración excesiva detectada en el eje secundario del equipo",
        "Imposible sincronizar los registros remotos mediante API",
        "Se requiere reemplazo urgente del filtro de aire primario",
        "La aplicación móvil se cierra inesperadamente al iniciar sesión"
    ],
    "label": [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
}

df = pd.DataFrame(raw_data)
# Multiplicamos el dataset para simular mayor volumen de pruebas
df = pd.concat([df] * 5, ignore_index=True)

ds = Dataset.from_pandas(df)
dataset_split = ds.train_test_split(test_size=0.3, seed=42)
dataset_dict = DatasetDict({
    'train': dataset_split['train'],
    'test': dataset_split['test']
})

print(f"Dataset cargado correctamente. Muestras de entrenamiento: {len(dataset_dict['train'])}, Muestras de evaluación: {len(dataset_dict['test'])}")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1)
    }


models_to_evaluate = [
    {"name": "BERT-Base-Multilingual", "hub_path": "bert-base-multilingual-cased"},
    {"name": "RoBERTa-Base", "hub_path": "roberta-base"},
    {"name": "DistilBERT-Base", "hub_path": "distilbert-base-uncased"}
]

benchmark_results = []

for m_info in models_to_evaluate:
    print(f"\n================ Processing: {m_info['name']} ================")
    model_name = m_info["hub_path"]

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

    # Preprocesamiento / Tokenización
    def tokenize_fn(examples):
        return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=64)

    tokenized_ds = dataset_dict.map(tokenize_fn, batched=True)

    # Entrenamiento mínimo de 1 época para ajustar los pesos del clasificador
    training_args = TrainingArguments(
        output_dir=f"./results_{m_info['name']}",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=1,
        logging_strategy="no",
        save_strategy="no",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds['train'],
        eval_dataset=tokenized_ds['test'],
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_res = trainer.evaluate()

    # Convertimos explícitamente a list(str) para el pipeline
    sample_texts = list(dataset_dict['test']['text'])
    classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0 if device=="cuda" else -1)

    start_time = time.time()
    _ = classifier(sample_texts)
    end_time = time.time()

    total_latency_sec = end_time - start_time
    avg_latency_ms = (total_latency_sec / len(sample_texts)) * 1000

    # Conteo de parámetros
    total_params_m = sum(p.numel() for p in model.parameters()) / 1e6

    benchmark_results.append({
        "Modelo": m_info["name"],
        "Parámetros (M)": round(total_params_m, 2),
        "Accuracy": round(eval_res["eval_accuracy"], 4),
        "Precision": round(eval_res["eval_precision"], 4),
        "Recall": round(eval_res["eval_recall"], 4),
        "F1-Score": round(eval_res["eval_f1"], 4),
        "Latencia (ms/muestra)": round(avg_latency_ms, 2)
    })


results_df = pd.DataFrame(benchmark_results)
print("\n=== RESUMEN COMPARATIVO DE RENDIMIENTO DE MODELOS ===")
print(results_df.to_string(index=False))

Dataset cargado correctamente. Muestras de entrenamiento: 56, Muestras de evaluación: 24

================ Processing: BERT-Base-Multilingual ================


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

... (31 outputs left out)

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Step,Training Loss


Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
No log,0.623151,7,0.833333,0.880952,0.833333,0.833333



=== RESUMEN COMPARATIVO DE RENDIMIENTO DE MODELOS ===
                Modelo  Parámetros (M)  Accuracy  Precision  Recall  F1-Score  Latencia (ms/muestra)
BERT-Base-Multilingual          177.85    1.0000     1.0000  1.0000    1.0000                  15.25
          RoBERTa-Base          124.65    0.4167     0.1736  0.4167    0.2451                  10.08
       DistilBERT-Base           66.96    0.8333     0.8810  0.8333    0.8333                   5.17
